# QLoRA (LoRA) fine-tuning — минимальный ноутбук
Этот ноутбук делает **QLoRA (4-bit)** дообучение (LoRA) для instruction-датасета формата JSONL.

**Подходит для GPU ~8GB** (RTX 3070 Ti Laptop).

## 0) Настройки
Поменяй `MODEL_NAME` на свою модель (7B обычно ок). Данные — `data/raw/train.jsonl` (instruction / input / output).

In [ ]:
import os

# Базовая instruct-модель (пример). Подставь свою.
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct")

# Путь к данным (JSONL)
DATA_PATH = os.getenv("DATA_PATH", "data/raw/train.jsonl")

# Куда сохранить LoRA-адаптер
OUT_DIR = os.getenv("OUT_DIR", "models/out_lora")

# Параметры под 8GB
MAX_SEQ_LEN = int(os.getenv("MAX_SEQ_LEN", "2048"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "1"))
GRAD_ACCUM = int(os.getenv("GRAD_ACCUM", "16"))
LR = float(os.getenv("LR", "2e-4"))
EPOCHS = float(os.getenv("EPOCHS", "1"))

print("MODEL_NAME:", MODEL_NAME)
print("DATA_PATH:", DATA_PATH)
print("OUT_DIR:", OUT_DIR)
print("MAX_SEQ_LEN:", MAX_SEQ_LEN, "BATCH_SIZE:", BATCH_SIZE, "GRAD_ACCUM:", GRAD_ACCUM, "LR:", LR, "EPOCHS:", EPOCHS)


## 1) Установка зависимостей
Если работаешь в контейнере/сервере — можно ставить один раз. На Windows чаще удобнее в WSL2/Linux или в Docker.

In [ ]:
# Если нужно — раскомментируй:
# !pip install -U "transformers>=4.41" "peft>=0.12" "accelerate>=0.30" datasets bitsandbytes trl


## 2) Проверка GPU / Torch

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 3) Подготовка данных
Ожидается JSONL, каждая строка:

```json
{"instruction":"...","input":"...","output":"..."}
```

Если файла нет — создадим демо на 5 строк.

In [ ]:
import json, pathlib

path = pathlib.Path(DATA_PATH)
path.parent.mkdir(parents=True, exist_ok=True)

if not path.exists():
    demo = [
        {"instruction": "Объясни, что такое католит.", "input": "", "output": "Католит — это ... (заполни своими данными)."},
        {"instruction": "Сформулируй кратко цель технологической инструкции.", "input": "", "output": "Цель инструкции — ..."},
        {"instruction": "Что делать, если в контексте нет информации?", "input": "", "output": "В контексте нет информации."},
        {"instruction": "Перечисли требования к безопасности кратко.", "input": "", "output": "Требования: ..."},
        {"instruction": "Сделай краткий ответ с ссылками.", "input": "Контекст: [doc#1] ...", "output": "Ответ ... [doc#1]"},
    ]
    with path.open("w", encoding="utf-8") as f:
        for row in demo:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("✅ Создан demo датасет:", path)
else:
    print("✅ Найден датасет:", path)

# Покажем 2 строки
with path.open("r", encoding="utf-8") as f:
    for _ in range(2):
        print(f.readline().strip())


## 4) Загрузка датасета и форматирование
Делаем простую разметку в текст: `Инструкция/Ввод/Ответ`.
Если у твоей модели есть официальный чат-шаблон — позже можно заменить на него.

In [ ]:
from datasets import load_dataset

def format_sample(ex):
    instr = (ex.get("instruction") or "").strip()
    inp = (ex.get("input") or "").strip()
    out = (ex.get("output") or "").strip()

    if inp:
        prompt = f"### Инструкция:\n{instr}\n\n### Ввод:\n{inp}\n\n### Ответ:\n"
    else:
        prompt = f"### Инструкция:\n{instr}\n\n### Ответ:\n"
    return {"text": prompt + out}

ds = load_dataset("json", data_files=str(path), split="train")
ds = ds.map(format_sample, remove_columns=ds.column_names)
print(ds[0]["text"][:400])
print("samples:", len(ds))


## 5) Загрузка модели в 4-bit и подключение LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True,  # QLoRA
)

model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Универсальный набор для многих LLaMA/Qwen-подобных моделей.
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


## 6) Тренировка (SFTTrainer)
Под 8GB: batch=1 и gradient_accumulation, max_seq_len 1024-2048.
Сохраняем адаптер в папку `OUT_DIR`.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tok,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=args,
)

trainer.train()

trainer.model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("✅ Saved LoRA to:", OUT_DIR)


## 7) Инференс: базовая модель + LoRA

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True,
)

lora_model = PeftModel.from_pretrained(base, OUT_DIR)
lora_model.eval()

def generate(prompt: str, max_new_tokens: int = 200):
    inputs = tok(prompt, return_tensors="pt")
    inputs = {k: v.to(lora_model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = lora_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    return tok.decode(out[0], skip_special_tokens=True)

test_prompt = "### Инструкция:\nОбъясни, что такое католит.\n\n### Ответ:\n"
print(generate(test_prompt, max_new_tokens=200))


## 8) Экспорт (опционально)
LoRA-адаптер уже сохранён в `OUT_DIR`. При желании можно слить (merge) LoRA в базовую модель, но для 4-bit это обычно делают в float16 на отдельной машине.